# DATA6000 Capstone — Rainfall vs Veg Production Forecast
## Google Colab Version
**Source file:** Final_Dataset_Filled.csv

Run each cell in order from top to bottom.

In [ ]:
# ── Install required libraries ────────────────────────────
!pip install matplotlib numpy scipy statsmodels scikit-learn --quiet
print("Libraries ready")

In [ ]:
# ── Load data from CSV ────────────────────────────────────
# Option 1: Upload CSV directly (recommended)
from google.colab import files
import io
import pandas as pd

print("Please upload your Final_Dataset_Filled.csv file")
uploaded = files.upload()
df = pd.read_csv(io.BytesIO(uploaded['Final_Dataset_Filled.csv']))

# Option 2: Mount Google Drive (uncomment if using Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/Final_Dataset_Filled.csv')

# Check required columns
required = ['Financial_Year', 'Total_Rainfall_mm', 'Total_Veg_Prod_t']
missing = [c for c in required if c not in df.columns]
if missing:
    print(f"WARNING: Missing columns: {missing}")
    print(f"Available columns: {list(df.columns)}")
else:
    print(f"Data loaded successfully: {df.shape[0]} rows")
    print(df[required].to_string(index=False))

# Make dataset available for the chart code
dataset = df

In [ ]:
# ============================================================
# DATA6000 Capstone — Rainfall vs Total Horticulture Forecast
# Power BI Python Visual Script
# Columns to drag: Financial_Year, Total_Rainfall_mm,
#                  Total_Production_Kt
# Shows: Historical + Forecast for both variables
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
from scipy import stats

df = dataset.copy()
df = df.sort_values('Financial_Year').reset_index(drop=True)
df = df.dropna(subset=['Total_Rainfall_mm', 'Total_Production_Kt'])

# ── Hardcoded Pearson R — do not change ──────────────────
PEARSON_R = -0.621
PEARSON_P = 0.2634

# ── Historical data ───────────────────────────────────────
hist_labels = list(df['Financial_Year'])
hist_rain   = list(df['Total_Rainfall_mm'])
hist_prod   = list(df['Total_Production_Kt'])
n_hist      = len(hist_labels)

# ── Forecast labels ───────────────────────────────────────
fore_labels = ['25/26', '26/27', '27/28', '28/29', '29/30']
n_fore      = len(fore_labels)

# ── Rainfall forecast — BOM declining trend ───────────────
# Linear slope from historical data
rain_slope, rain_intercept, *_ = stats.linregress(
    np.arange(n_hist), hist_rain)
fore_rain = [max(800, rain_intercept +
             rain_slope * (n_hist + i))
             for i in range(n_fore)]

# ── Production forecast — post-changepoint slope ─────────
# Using Holt-Winters style dampened decline
prod_last   = hist_prod[-1]
prod_peak   = max(hist_prod)
post_slope  = (prod_last - prod_peak) * 0.25
fore_prod   = [prod_last + post_slope * (i + 1)
               for i in range(n_fore)]

# ── All labels combined ───────────────────────────────────
all_labels = hist_labels + fore_labels
x_all      = np.arange(len(all_labels))
x_hist     = x_all[:n_hist]
x_fore     = x_all[n_hist:]
mid        = (x_hist[-1] + x_fore[0]) / 2

# ── Colours ───────────────────────────────────────────────
C_BG       = '#F8FAF8'
C_RAIN     = '#185FA5'
C_RAIN_F   = '#94A3B8'
C_PROD     = '#1E6B2E'
C_PROD_F   = '#E24B4A'
C_GRID     = '#E5E5E5'
C_TITLE    = '#0B1F12'
C_MUTED    = '#64748B'

# ── Figure ────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(C_BG)
ax1.set_facecolor('white')

# ── Rainfall bars — historical ────────────────────────────
ax1.bar(x_hist, hist_rain,
        width=0.38, color=C_RAIN,
        alpha=0.65, zorder=2,
        label='Rainfall — Historical (mm)')

# ── Rainfall bars — forecast ──────────────────────────────
ax1.bar(x_fore, fore_rain,
        width=0.38, color=C_RAIN_F,
        alpha=0.55, zorder=2,
        label='Rainfall — Forecast (mm)')

# ── Rainfall value labels ─────────────────────────────────
for i, val in enumerate(hist_rain):
    ax1.text(x_hist[i], val + 35,
             f'{val:,.0f}mm',
             ha='center', fontsize=7.5,
             color=C_RAIN, fontweight='bold')
for i, val in enumerate(fore_rain):
    ax1.text(x_fore[i], val + 35,
             f'{val:,.0f}mm',
             ha='center', fontsize=7.5,
             color=C_RAIN_F, fontweight='bold')

# ── Rainfall axis ─────────────────────────────────────────
ax1.set_ylabel('Total Rainfall (mm)',
               fontsize=10, color=C_RAIN, labelpad=10)
ax1.tick_params(axis='y', colors=C_RAIN, labelsize=9)
ax1.set_ylim(0, max(hist_rain + fore_rain) * 1.45)

# ── Production line — right axis ──────────────────────────
ax2 = ax1.twinx()

# Historical production
ax2.plot(x_hist, hist_prod,
         color=C_PROD, linewidth=2.8,
         marker='o', markersize=8,
         markerfacecolor=C_PROD,
         markeredgecolor='white',
         markeredgewidth=2.2,
         label='Production — Historical (K t)',
         zorder=6)

# Dotted connector
ax2.plot([x_hist[-1], x_fore[0]],
         [hist_prod[-1], fore_prod[0]],
         color=C_PROD_F, linewidth=1.5,
         linestyle=':', alpha=0.5, zorder=4)

# Forecast production
ax2.plot(x_fore, fore_prod,
         color=C_PROD_F, linewidth=2.5,
         marker='D', markersize=7,
         linestyle='--',
         markerfacecolor=C_PROD_F,
         markeredgecolor='white',
         markeredgewidth=2,
         label='Production — Forecast (K t)',
         zorder=6)

# ── Production value labels ───────────────────────────────
prod_all = hist_prod + fore_prod
ymin_p   = min(prod_all) * 0.975
ymax_p   = max(prod_all) * 1.055

for i, val in enumerate(hist_prod):
    ax2.text(x_hist[i],
             val + (ymax_p - ymin_p) * 0.018,
             f'{val:.0f}K',
             ha='center', fontsize=8,
             color=C_PROD, fontweight='bold')
for i, val in enumerate(fore_prod):
    ax2.text(x_fore[i],
             val + (ymax_p - ymin_p) * 0.018,
             f'{val:.0f}K',
             ha='center', fontsize=8,
             color=C_PROD_F, fontweight='bold')

ax2.set_ylabel('Total Horticulture Production (K tonnes)',
               fontsize=10, color=C_PROD, labelpad=10)
ax2.tick_params(axis='y', colors=C_PROD, labelsize=9)
ax2.set_ylim(ymin_p, ymax_p)

# ── Highlight irrigation buffer collapse ──────────────────
collapse_idx = n_hist - 1
ax2.annotate(
    'Buffer\ncollapse',
    (x_hist[collapse_idx], hist_prod[collapse_idx]),
    textcoords='offset points',
    xytext=(-45, -40),
    ha='center', fontsize=8,
    color='#E24B4A', fontweight='bold',
    arrowprops=dict(arrowstyle='->',
                    color='#E24B4A', lw=1.2))

# ── Highlight peak production ─────────────────────────────
peak_idx = int(np.argmax(hist_prod))
ax2.scatter(x_hist[peak_idx],
            hist_prod[peak_idx],
            color='#BA7517', s=160, zorder=8,
            edgecolors='white', linewidth=2.5)
ax2.annotate(
    f'Peak\n{hist_prod[peak_idx]:.0f}K t',
    (x_hist[peak_idx], hist_prod[peak_idx]),
    textcoords='offset points',
    xytext=(0, 18),
    ha='center', fontsize=8,
    color='#BA7517', fontweight='bold',
    arrowprops=dict(arrowstyle='-',
                    color='#BA7517', lw=1.2))

# ── Divider line ──────────────────────────────────────────
ax1.axvline(x=mid, color='#94A3B8',
            linestyle='--', linewidth=1.2,
            alpha=0.5, zorder=3)
ax1.text(mid - 1.2,
         max(hist_rain + fore_rain) * 1.35,
         'Historical',
         ha='center', fontsize=9,
         color='#94A3B8', style='italic',
         fontweight='bold')
ax1.text(mid + 1.5,
         max(hist_rain + fore_rain) * 1.35,
         'Forecast \u2192',
         ha='center', fontsize=9,
         color=C_PROD_F, style='italic',
         fontweight='bold')

# ── Pearson R annotation ──────────────────────────────────
ax1.text(0.01, 0.06,
         f'Pearson R = {PEARSON_R}  '
         f'(p = {PEARSON_P})\n'
         f'Negative R reflects irrigation dependency\n'
         f'not a direct inverse relationship',
         transform=ax1.transAxes,
         fontsize=8.5, color=C_MUTED,
         bbox=dict(boxstyle='round,pad=0.5',
                   facecolor='white',
                   edgecolor='#E2E8F0',
                   alpha=0.92))

# ── Axes formatting ───────────────────────────────────────
ax1.set_xticks(x_all)
ax1.set_xticklabels(all_labels,
                    fontsize=9, color=C_MUTED)
ax1.set_xlim(-0.6, len(all_labels) - 0.4)
ax1.set_xlabel('Financial Year',
               fontsize=10, color=C_MUTED,
               labelpad=8)
ax1.tick_params(axis='x', colors=C_MUTED,
                labelsize=9, length=0)
ax1.tick_params(axis='y', length=0)
ax1.yaxis.grid(True, color=C_GRID,
               linewidth=0.8, linestyle='-')
ax1.xaxis.grid(False)
ax1.set_axisbelow(True)
ax1.spines['top'].set_visible(False)
ax2.spines['top'].set_visible(False)

# ── Title ─────────────────────────────────────────────────
ax1.set_title(
    'SA Rainfall vs Total Horticulture Production \u2014 '
    'Historical & Forecast (FY 20/21\u201329/30)',
    fontsize=13, fontweight='bold',
    color=C_TITLE, pad=14)
ax1.text(0.0, 1.025,
         f'Pearson R = {PEARSON_R}  (irrigation buffer effect)  '
         f'\u00b7  Forecast: BOM declining rainfall + '
         f'post-changepoint production trend',
         transform=ax1.transAxes,
         fontsize=8.5, color=C_MUTED,
         style='italic')

# ── Legend ────────────────────────────────────────────────
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
peak_p  = mpatches.Patch(
    color='#BA7517',
    label='Production peak (FY 23/24)')
ax1.legend(
    h1 + h2 + [peak_p],
    l1 + l2 + ['Production peak (FY 23/24)'],
    fontsize=8.5, loc='upper right',
    framealpha=0.92,
    edgecolor='#E2E8F0',
    fancybox=False)

plt.tight_layout()
plt.savefig('Rainfall_vs_Production_Forecast.png',
            dpi=180, bbox_inches='tight',
            facecolor=C_BG)

from IPython.display import Image, display
display(Image('Rainfall_vs_Production_Forecast.png'))
print("Saved: Rainfall_vs_Production_Forecast.png")
